# AAI6640 2026 Spring Assignment 4

## Section One: Conceptual Understanding

## <div class="alert alert-info">[GRADED TASK 1.1]</div>

Explain BPTT and show the process of updating $W_y, W_h, W_x$

# Your answer here

BPTT (Backpropagation Through Time ) is the primarily used for RNN (recurrence neural network) because sequential data processing relies on correct input order and not just the standard backpropagation where the same weights are reused at each step. 
BPTT  handles this in 4 phases 
1) Unrolling – takes a single cell ‘h(t)’ and unrolls across T time steps (e.g. T=3 steps)
    Its rolled out view, shows a representation of a feedforward network. With one copy of ‘h(t)’ per step 

Represented as:
h(0) -> [STEP 1: Wx, Wh, Wy] -> h(1) -> [STEP 2: Wx, Wh, Wy] -> h(2) -> [STEP 3: Wx, Wh, Wy] -> ŷ(3)
        ↑			                        ↑				                ↑
        x(1)					        x(2)				               x(3)
2) Forward Pass 
This will be used to compute through each of the T steps until the predicted outputs are reached ŷ(T)
3) Compute for Loss at the output ŷ(T) to handle regression 
4) Backward Pass 
Working backwards from the goal to start, we will be able to propagate gradients at each time step. We can then understand how much each step contributed to the loss by accumulating their contributions from all the steps. 

To apply activation function RNN work well with tanh as its default. It can be represented in the equation as:
h(t) = tanh(Wx * x(t) + Wh * h(t-1) + bh) 

where h(t): the # of unrolled steps 
tanh: is the activation function which squashes everything from a range [-1,1] 
(Wx * x(t) + Wh * h(t-1) + bh): 
    Wx * x(t): takes the current position and transforms it
    Wh * h(t-1):  takes the previous hidden state and compute it against  the current step
    Bh: handles the bias term, prevents the model from zeroing. Position at each layer

The second core equation: ŷ(t) = Wy · h(t) + by
ŷ(t): is the models prediction output, which will be used to compare against the true actual labels to compute the loss.
Wy · h(t):  maps through the hidden space to output size 
by:  the bias for the output layer
---------------------------------------------------------------

Updating Wy 
Only seen within the output equation which signifies the actual true values of the output. It does not show up with the recurrence relation. Since gradient only passes through loss to prediction to output at this stage, this is recognized as a straightforward pass that uses standard backprop. 

It also does not suffer from vanishing gradients, so no time-rolling is needed 
dL/dWy = (dL/dŷ(t)) * (dŷ(t)/dWy)
---------------------------------------------------------------
Updating Wh
In the unrolling process, Wh appears at each of the recurrence. Gradients propagate through all paths and sum during backpropagation.
For a simple setup, we can use a 3-step where T=3 and unrolls into 3 paths:
Path 1 (k=3): dL/dŷ(3) * dŷ(3)/dh(3)  * dh(3)/dWh – with 0 Jacobians
Path 2 (k=2): dL/dŷ(3) * dŷ(3)/dh(3)  * dh(3)/dh(2) * dh(2) /dWh – with 1Jacobian
Path 3 (k=1): dL/dŷ(3) * dŷ(3)/dh(3)  * dh(3)/dh(2) * dh(2) /dh(1) * dh(1) /dWh – with 2 Jacobians

Each Jacobian involves tanh which is between 0 and 1. As the steps are tested against each other, the values continue to shrink exponentially, eventually leading to the issue of vanishing gradients. Early steps tend to almost be stagnant and do nothing, while recent steps continue to dominate. Each Jacobian is represented as dh(i) / dh(i-1) (takes the previous hidden state and compute it against  the current)
------------------------------------------------------
Updating Wx
Wx also appears at every step in the recurrence, the structure is like Wh where they both have the same sum-of-paths format. 
It suffers from the same vanishing/exploding gradients issues when values get too small



# Section Two - Coding Practice

In [2]:
# please do not modify this cell
# ---------------------------
# Temperature
# ---------------------------
temp_data = [
[1949,34.7,34.4,39.2,50.9,60.3,71.6,76.3,74.4,63.2,58.4,43.5,36.8],
[1950,36.2,28.0,33.7,46.7,55.8,69.1,73.5,70.8,61.5,56.3,47.8,35.5],
[1951,34.0,34.7,39.0,51.0,59.1,66.0,74.0,70.6,65.5,54.8,42.6,35.0],
[1952,32.6,32.5,37.2,50.5,57.2,70.6,77.5,72.2,66.3,53.0,44.9,35.6],
[1953,34.7,35.0,39.0,48.9,58.4,70.5,73.2,72.0,66.3,56.2,48.6,40.1],
[1954,25.9,36.4,38.8,50.1,56.3,66.6,72.1,70.1,63.2,58.6,44.4,34.4],
[1955,28.5,32.0,37.6,49.2,62.7,66.8,77.2,74.5,64.5,55.1,41.9,26.6],
[1956,30.6,32.5,33.5,45.6,55.4,68.9,71.7,71.8,61.1,54.3,46.1,36.0],
[1957,23.4,34.6,39.0,49.4,59.8,71.3,74.1,69.3,67.3,54.5,47.1,40.0],
[1958,31.0,25.5,39.0,48.7,56.6,63.9,72.4,72.4,64.6,52.6,46.5,26.4],
[1959,28.7,26.7,37.0,48.9,62.6,64.6,74.7,74.1,68.1,55.0,44.4,36.3],
[1960,30.9,35.3,32.7,48.3,59.7,69.6,73.1,72.2,63.7,53.9,48.0,29.5],
]
columns = ["Year","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# ---------------------------
# CPI
# ---------------------------
cpi_data = [
[1949,24.0,23.8,23.8,23.9,23.8,23.9,23.7,23.8,23.9,23.7,23.8,23.6],
[1950,23.5,23.5,23.6,23.6,23.7,23.8,24.1,24.3,24.4,24.6,24.7,25.0],
[1951,25.4,25.7,25.8,25.8,25.9,25.9,25.9,25.9,26.1,26.2,26.4,26.5],
[1952,26.5,26.3,26.3,26.4,26.4,26.5,26.7,26.7,26.7,26.7,26.7,26.7],
[1953,26.6,26.5,26.6,26.6,26.7,26.8,26.8,26.9,26.9,27.0,26.9,26.9],
[1954,26.9,26.9,26.9,26.8,26.9,26.9,26.9,26.9,26.8,26.8,26.8,26.7],
[1955,26.7,26.7,26.7,26.7,26.7,26.7,26.8,26.8,26.9,26.9,26.9,26.8],
[1956,26.8,26.8,26.8,26.9,27.0,27.2,27.4,27.3,27.4,27.5,27.5,27.6],
[1957,27.6,27.7,27.8,27.9,28.0,28.1,28.3,28.3,28.3,28.3,28.4,28.4],
[1958,28.6,28.6,28.8,28.9,28.9,28.9,29.0,28.9,28.9,28.9,29.0,28.9],
[1959,29.0,28.9,28.9,29.0,29.0,29.1,29.2,29.2,29.3,29.4,29.4,29.4],
[1960,29.3,29.4,29.4,29.5,29.5,29.6,29.6,29.6,29.6,29.8,29.8,29.8],
]

## <div class="alert alert-info">[GRADED TASK 2.1]</div>
Preparing data:
- Load the original data from AirPassengers.csv.
- Combine it with the given temperature and CPI data.
- Use the last three years as the test set.  

In [3]:
# Your answer here

#-------- IMPORTS -------

# handle data manipulation and operations 
import numpy as np
import pandas as pd 

# torch for deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset

# Visualization and plotting 
import matplotlib.pyplot as plt

# skilearn for data preprocessing and evaluation metrics 
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

# will need to compute the absolute mean error as a percentage on the target variables. Will be used for RNN and LSTM
def mean_absolute_percentage_error(y_true, y_pred):
    # setting up format of the function as numpy array
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    
    # subtracting the true values from prediected, then divide by the  true values; take the absolute value and mean and times it by 100 to get a percentage
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


print("Imports loaded successfully")
print("=" * 50)
print("MAPE function defined successfully")
print(f"Mean Absolute Percentage Error Function Test: {mean_absolute_percentage_error([100, 200, 300], [110, 190, 310]):.2f} %")

Imports loaded successfully
MAPE function defined successfully
Mean Absolute Percentage Error Function Test: 6.11 %


In [4]:
# load the csv dataset into pandas dataframe 
air_passengers_df = pd.read_csv("AirPassengers.csv")

# split the dataset in the month column (year-month to year and month)
air_passengers_df[['Year', 'Month']] = air_passengers_df['Month'].str.split('-', expand=True)

# rearrange the columns 
air_passengers_df = air_passengers_df[['Year', 'Month', 'Passengers']]
air_passengers_df['Year'] = air_passengers_df['Year'].astype(int)
air_passengers_df['Month'] = air_passengers_df['Month'].astype(int)




print("Dataset loaded successfully")
print("=" * 50)
print(f"Dataset shape: {air_passengers_df.shape}")
print(f"Dataset columns: {air_passengers_df.columns.tolist()}")
print("-" * 50)
print(f"Dataset head:\n{air_passengers_df.head().to_string(index=False)}")


Dataset loaded successfully
Dataset shape: (144, 3)
Dataset columns: ['Year', 'Month', 'Passengers']
--------------------------------------------------
Dataset head:
 Year  Month  Passengers
 1949      1         112
 1949      2         118
 1949      3         132
 1949      4         129
 1949      5         121


In [5]:
#---- Turn cpi, temp into dataframe and merge with air_passengers_df

# convert list to dataframe (temp and cpi)
temp_df = pd.DataFrame(temp_data, columns=columns)
cpi_df = pd.DataFrame(cpi_data, columns=columns)

# convert the dataframe from wide to long format to match the structure of air_passengers_df

# map the month names to their corresponding number. Ex Jan to 01, Feb to 02, etc
month_mapping = {
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12
}

'''columns=columns gives the Dataframe the column headers. melt unpivots the columns into rows
and dumps the headers into the new column name var_name='MonthName'

now the dataframe has two of the same columns MonthName and Month
then .map() month to the dictionary month+mapping to get the month number and assign it to the new Month column.

Since the month column is a string, we convert those values to integers.

now MonthName and Month are two different types and we only need the Month column (int) so we drop the MonthName text value 

reference: https://pandas.pydata.org/docs/reference/api/pandas.melt.html
'''
# melt the dataframe from wide to long format 
# ---- temp_df melt -------
temp_df = temp_df.melt(id_vars="Year", var_name="MonthName", value_name="Temperature")
temp_df['Month'] = temp_df['MonthName'].map(month_mapping) # map the month name to their corresponding number
temp_df['Month'] = temp_df['Month'].astype(int) # convert month column to integer
temp_df = temp_df.drop(columns=['MonthName']) # drop MonthName column since we have the month number and do not need the text value

# sort dataframe by Year and Month 
temp_df = temp_df.sort_values(by=['Year', 'Month']).reset_index(drop=True)

''' After testing the melt grouped all the Jans, Febs, etc together. The values need sorting to match correct structure'''
# test temp_df after melt and mapping to check for proper output and format
print("Temperature melt and mapped successfully")
print("~" * 50)
print(temp_df.to_string(index=False))

# ------- cpi_df melt -------
cpi_df = cpi_df.melt(id_vars="Year", var_name="MonthName", value_name="CPI")
cpi_df['Month'] = cpi_df['MonthName'].map(month_mapping) # map the month name to their corresponding number
cpi_df['Month'] = cpi_df['Month'].astype(int) # convert month column to integer
cpi_df = cpi_df.drop(columns=['MonthName']) # drop MonthName column since we have the month number and do not need the text value

# sort dataframe by Year and Month 
cpi_df = cpi_df.sort_values(by=['Year', 'Month']).reset_index(drop=True)

# test cpi_df after melt and mapping to check for proper output and format
print("=" * 50)
print("CPI melt and mapped successfully\n")
print("~" * 50)
print(cpi_df.to_string(index=False))




Temperature melt and mapped successfully
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
 Year  Temperature  Month
 1949         34.7      1
 1949         34.4      2
 1949         39.2      3
 1949         50.9      4
 1949         60.3      5
 1949         71.6      6
 1949         76.3      7
 1949         74.4      8
 1949         63.2      9
 1949         58.4     10
 1949         43.5     11
 1949         36.8     12
 1950         36.2      1
 1950         28.0      2
 1950         33.7      3
 1950         46.7      4
 1950         55.8      5
 1950         69.1      6
 1950         73.5      7
 1950         70.8      8
 1950         61.5      9
 1950         56.3     10
 1950         47.8     11
 1950         35.5     12
 1951         34.0      1
 1951         34.7      2
 1951         39.0      3
 1951         51.0      4
 1951         59.1      5
 1951         66.0      6
 1951         74.0      7
 1951         70.6      8
 1951         65.5      9
 1951         54.8     1

In [6]:
# ---- merge the dataframes --------
merge_df = air_passengers_df[[ 'Year', 'Month', 'Passengers']].merge(temp_df, on=['Year', 'Month'], how="left").merge(cpi_df, on=['Year', 'Month'], how="left")
print("Dataframes merged successfully")
print(f"Merged Dataframe: \n{merge_df.to_string(index=False, col_space={'Month': 10, 'Passengers': 15, 'Temperature': 15, 'CPI': 10})}")

Dataframes merged successfully
Merged Dataframe: 
 Year      Month      Passengers     Temperature        CPI
 1949          1             112            34.7       24.0
 1949          2             118            34.4       23.8
 1949          3             132            39.2       23.8
 1949          4             129            50.9       23.9
 1949          5             121            60.3       23.8
 1949          6             135            71.6       23.9
 1949          7             148            76.3       23.7
 1949          8             148            74.4       23.8
 1949          9             136            63.2       23.9
 1949         10             119            58.4       23.7
 1949         11             104            43.5       23.8
 1949         12             118            36.8       23.6
 1950          1             115            36.2       23.5
 1950          2             126            28.0       23.5
 1950          3             141            33.7  

In [7]:
# get the last 3 years as a test set and the rest as a training set 
# we are using the merge dataframe from above 
# test set 
test_df = merge_df[merge_df['Year'] >= 1958]
# training set
train_df = merge_df[merge_df['Year'] <= 1957]

print("=" * 50)
print("Train and Test sets created successfully")
print("~" * 50)
print(f"Train set shape: {train_df.shape[0]} rows, {train_df.shape[1]} columns ")
print(f"Test set shape: {test_df.shape[0]} rows, {test_df.shape[1]} columns ")

#print(test_df.to_string(index=False))
#print(train_df.to_string(index=False))

Train and Test sets created successfully
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Train set shape: 108 rows, 5 columns 
Test set shape: 36 rows, 5 columns 


In [ ]:
'''
Before we can build the model we need to address preprocessing phase

This section will implement BPTT by:
- normalizing: will use MinMaxScaler to scale features in range of [0,1] 
- creating window sequences for RNN input 
- converting to Pytorch tensors ( tensors are data structures that deep learning models can process)
'''


# set features to be fit and scaled by MinMaxScaler
features = ["Passengers", "Temperature", "CPI"]

# initialize the scaler 
scaler = MinMaxScaler()

# fit the scaler on the training data and transform from test and train sets
scaler.fit(train_df[features])

# transform the features 
trained_scaled = scaler.transform(train_df[features])
test_scaled = scaler.transform(test_df[features])


# test scaled output 
print("Data scaled successfully")
print(f"Train scaled shape: {trained_scaled.shape}")
print(f"Test scaled shape: {test_scaled.shape}")
print(f"min: {trained_scaled.min():.2f}, max: {trained_scaled.max():.2f}")


# ----- Preparing a rolling winddow dataset -----
'''
grabbing a chunk of data (window) = X (input features) then we grab the 'y' (the target) the next value in the sequence.

reference: 
https://docs.pytorch.org/docs/stable/generated/torch.tensor.html
https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset

'''

def create_dataset(data, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:(i + window)])
        y.append(data[i + window, 0]) # after testing y.append(data[i + window])  grabs all the features. only need the first column (Passengers)
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

# set window size - default is 12 months (1 year)
window_size = 12

# call the create_dataset function to create the dataset for test and train sets
X_train, y_train = create_dataset(trained_scaled, window_size)
X_test, y_test = create_dataset(test_scaled, window_size)

# test the output shapes of X and y 
print("=" * 50)
print("Rolling window dataset created successfully\n")
print(f"X train shape: {X_train.shape}") # 96 samples, 12 time steps, 3 features (passengers, temp, cpi)
print(f"y train shape: {y_train.shape}")
print(f"X test shape: {X_test.shape}")
print(f"y test shape: {y_test.shape}")

#X_train, y_train

Data scaled successfully
Train scaled shape: (108, 3)
Test scaled shape: (36, 3)
min: 0.00, max: 1.00
Rolling window dataset created successfully

X train shape: torch.Size([96, 12, 3])
y train shape: torch.Size([96])
X test shape: torch.Size([24, 12, 3])
y test shape: torch.Size([24])


/var/folders/n4/hv8fmd6144d48b1sx732llg80000gn/T/ipykernel_59919/1734265163.py:47: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


## <div class="alert alert-info">[GRADED TASK 2.2]</div>
Building and comparing two predictive models—RNN and LSTM—using the following metrics:
- $R^2$
- Root Mean Squared Error (RMSE)
- Mean Absolute Percentage Error (MAPE)

In [11]:
'''
instead of using SimpleNN, we can use Pytorch built in RNN and LSTM module 


Findings: 
- calling nn.RNN() you do not have to call initialize tanh() as its built in as the default activation function

References: 
https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html
https://discuss.pytorch.org/t/could-someone-explain-batch-first-true-in-lstm/15402
https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html
https://docs.pytorch.org/docs/stable/tensors.html#torch.Tensor
https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html

Module 12 Notes - "Building RNNs with Pytorch"

'''

#-------- Build of the RNN Model --------
class RNNModel(nn.Module):
    def __init__(self, input_size=3, hidden_size=32, num_layers=1, output_size=1):
        # calling the super class to initialize the nn.Module which is the parent class of RNNModel
        super(RNNModel, self).__init__()
        
        # initalize the hidden_size, num_layers 
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # define the RNN layer, calling RNN API from Pytorch
        self.rnn = nn.RNN(input_size = input_size, hidden_size = hidden_size, num_layers = num_layers, batch_first=True)
        
        # setup unrolled recurrence to output layer
        self.fc = nn.Linear(hidden_size, output_size)
        
    # implement forward pass 
    def forward(self, x): 
        # initialize the starting hidden state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        # pass the input through the RNN layer, get ther ouput and hidden state
        out, _ = self.rnn(x, h0) # dropping the hidden state output because its not needed, only need output of final time step
        
        out = self.fc(out[:, -1, :]) # using array slicing through all hidden values to get the last time step
        
        return out 


print("RNN class defined and compiled successfully")
print("=" * 50)
print("Testing RNNModel class:\n")
# create an instance of the RNNModel class
model_rnn = RNNModel()
# test on X_train to check for proper output 
test_output = model_rnn(X_train[:5]) # testing the first 5 samples of the training set
print(f"Test input shape: {X_train[:5].shape}")
print(f"Test output shape: {test_output.shape}")


 #-------- Build of the LSTM Model --------
class LSTMModel(nn.Module):
    def __init__(self, input_size=3, hidden_size=32, num_layers=1, output_size=1):
        # calling the super class to initialize the nn.Module which is the parent class of LSTMModel
        super(LSTMModel, self).__init__()
        
        # initalize the hidden_size, num_layers 
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # define the LSTM layer, calling LSTM API from Pytorch
        self.lstm = nn.LSTM(input_size = input_size, hidden_size = hidden_size, num_layers = num_layers, batch_first=True)
        
        # setup unrolled recurrence to output layer
        self.fc = nn.Linear(hidden_size, output_size)
        
    # implement forward pass 
    def forward(self, x): 
        # initialize the starting hidden state with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        # pass the input through the LSTM layer, get the output and hidden state
        out, _ = self.lstm(x, (h0, c0)) # 
        
        out = self.fc(out[:, -1, :]) # using array slicing through all hidden values to get the last time step
        
        return out 

print("~" * 50)
print("LSTM class defined and compiled successfully")
print("=" * 50)
print("Testing LSTMModel class:\n")
# create an instance of the LSTMModel class
model_lstm = LSTMModel()
# test on X_train to check for proper output 
test_output = model_lstm(X_train[:5]) # testing the first 5 samples of the training set
print(f"Test input shape: {X_train[:5].shape}")
print(f"Test output shape: {test_output.shape}")
        

       
        
        

RNN class defined and compiled successfully
Testing RNNModel class:

Test input shape: torch.Size([5, 12, 3])
Test output shape: torch.Size([5, 1])
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
LSTM class defined and compiled successfully
Testing LSTMModel class:

Test input shape: torch.Size([5, 12, 3])
Test output shape: torch.Size([5, 1])


In [ ]:
''' This section will be used for training and evaluation. 
For conistency, I will reuse the same training and evalaution techniques

Will use one training function for both RNN and LSTM models
since both have same structure and output 
'''

# ----------- TRAIN THE MODEL -----------
def model_train(model, X_train, y_train, epochs, lr=0.01):    
    # set the model to training mode
    model.train()
     
    # get the loss function for MSE using nn.MSELoss()
    criterion = nn.MSELoss()
    
    # get the optimzer for the model. Using Adam optimizer, PyTorch does not have a default optimizer
    optimizer = optim.Adam(model.parameters(), lr=lr)
        
    # iterate over the training data at each epoch
    for epoch in range(epochs):
    
            
        # optimize the model weights 
        optimizer.zero_grad()          # zero the gradients before backpropagation, clear old gradients
            
        # output the model
        output = model(X_train).squeeze() # squeeze to remove extra dimensions and match the shape of y_train
            
        # compute MSE loss between model output and true labels from y_train set
        loss = criterion(output, y_train)
            
        # optimize the model weights 
        loss.backward()                # compute gradients based on the loss
           
        # update the models weights based on gradient compute and optimization
        optimizer.step()
     
        # print the epoch at every 20 epochs to avoid too much output
        if epoch % 20 == 0:
            print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')
            
    # return the model after training is complete
    return model     

# return the model and final loss 
print('Training function has been defined successfully')
print("=" * 50)




# ----------- EVALUATE THE MODEL -----------
    
def evaluate_model(model, X_test, y_test):
            
    
    # activate evaluation mode for the model
    model.eval()
    
    # compute predictions without computing gradients since we are in evalaution mode
    with torch.no_grad():
       predictions = model(X_test).squeeze() # removing extra dimensions to match the y_test shape 

    # convert Pytorch tensors predictions to numpy arrays MSE and r2_score are sklearn functions and require numpy arrays
    predictions = predictions.numpy()
    
    # do the same for actual labels, which is y_test
    actual_labels = y_test.numpy()
    
    # computer metrics r2_score, MSE, and MAPE
    r2 = r2_score(actual_labels, predictions)
    rmse = np.sqrt(mean_squared_error(actual_labels, predictions))
    mape = mean_absolute_percentage_error(actual_labels, predictions)

    # print the evaluation metrics
    print(f"R2 Score: {r2:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAPE: {mape:.4f}")
    
    # store and return results for later use 
    eval_results = {
        "R2 Score": r2,
        "RMSE": rmse,
        "MAPE": mape
    }
    return eval_results
print("The evaluation function is defined successfully!")
    
    
    

Training function has been defined successfully
The evaluation function is defined successfully!


In [16]:
'''Testing Train and Evaluate functions'''


# ============================TEST MODEL ON TRAIN AND EVALUATE======================================================
# test the model_train_one_epoch and evaluate_model functions 
rnn_tester_model = RNNModel(input_size=3, hidden_size=32, num_layers=1, output_size=1) 
lstm_tester_model = LSTMModel(input_size=3, hidden_size=32, num_layers=1, output_size=1) 
print("Testing models against Training function:\n")
#training for 1 epoch
print("Now Testing RNN Training:")
print("~" * 30)
tester_rnn_output = model_train(rnn_tester_model, X_train, y_train, epochs=200, lr=0.01)
tester_rnn_output
print("\nNow Testing LSTM Training:")
print("~" * 30)
tester_lstm_output = model_train(lstm_tester_model, X_train, y_train, epochs=200, lr=0.01) 
tester_lstm_output

print("=" * 80)
print("Evaluation on the models:\n")
print("Now Testing RNN Evaluation:")
rnn_results_tester = evaluate_model(tester_rnn_output, X_test, y_test)
print("\nNow Testing LSTM Evaluation:")
lstm_results_tester = evaluate_model(tester_lstm_output, X_test, y_test)


Testing models against Training function:

Now Testing RNN Training:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Epoch 1/200, Loss: 0.3310
Epoch 21/200, Loss: 0.0126
Epoch 41/200, Loss: 0.0058
Epoch 61/200, Loss: 0.0036
Epoch 81/200, Loss: 0.0028
Epoch 101/200, Loss: 0.0023
Epoch 121/200, Loss: 0.0021
Epoch 141/200, Loss: 0.0019
Epoch 161/200, Loss: 0.0018
Epoch 181/200, Loss: 0.0015

Now Testing LSTM Training:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Epoch 1/200, Loss: 0.2740
Epoch 21/200, Loss: 0.0310
Epoch 41/200, Loss: 0.0109
Epoch 61/200, Loss: 0.0064
Epoch 81/200, Loss: 0.0044
Epoch 101/200, Loss: 0.0035
Epoch 121/200, Loss: 0.0029
Epoch 141/200, Loss: 0.0029
Epoch 161/200, Loss: 0.0023
Epoch 181/200, Loss: 0.0019
Evaluation on the models:

Now Testing RNN Evaluation:
R2 Score: 0.7717
MSE: 0.0097
MAPE: 7.9173

Now Testing LSTM Evaluation:
R2 Score: 0.7643
MSE: 0.0100
MAPE: 9.0225


In [17]:
'''
Created tester models to test the training and evaluation functions.

This section will train and evaluate the RNN and LSTM models against the full dataset and compare the results. 

Key takeaways from the TESTER phase:
- Both models trained with no errors.
    - had to change the epochs from 30 to 200 because r2_score was returning negative values, 
    indicating the 30 epochs wasnt enough and the models worse than just guessing.
- LSTM performed better than RNN on  all 3 metrics on every run of the tester phase.
- LSTM had more variance in passenger counts 
- evaluation metrics were decent for both models but LSTM had better across the board 
    - resulting in predictions being closer to the actual values 
- LSTM handles sequential data better than RNN and avoids vanishing gradient issuses commonly seen in RNNs
    - allows for long term dependencies to capture and learn seasonal patterns in the data

'''


# ---- Train and Evaluate the RNN Model ----

# initialize the RNN model
rnn_model = RNNModel()

print("Training the RNN model:")
print("~" * 30)
rnn_model = model_train(rnn_model, X_train, y_train, epochs=200, lr=0.01)
print("=" * 50)
print("=" * 50)
print("Evaluating the RNN model:")
rnn_results = evaluate_model(rnn_model, X_test, y_test)


# initialize the LSTM model
lstm_model = LSTMModel()

print("Training the LSTM model:")
print("~" * 30)
lstm_model = model_train(lstm_model, X_train, y_train, epochs=200, lr=0.01)
print("=" * 50)
print("=" * 50)
print("Evaluating the LSTM model:")
lstm_results = evaluate_model(lstm_model, X_test, y_test)




Training the RNN model:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Epoch 1/200, Loss: 0.6807
Epoch 21/200, Loss: 0.0275
Epoch 41/200, Loss: 0.0075
Epoch 61/200, Loss: 0.0058
Epoch 81/200, Loss: 0.0044
Epoch 101/200, Loss: 0.0032
Epoch 121/200, Loss: 0.0026
Epoch 141/200, Loss: 0.0021
Epoch 161/200, Loss: 0.0044
Epoch 181/200, Loss: 0.0027
Evaluating the RNN model:
R2 Score: 0.5892
MSE: 0.0174
MAPE: 9.2863
Training the LSTM model:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Epoch 1/200, Loss: 0.1715
Epoch 21/200, Loss: 0.0121
Epoch 41/200, Loss: 0.0069
Epoch 61/200, Loss: 0.0037
Epoch 81/200, Loss: 0.0030
Epoch 101/200, Loss: 0.0026
Epoch 121/200, Loss: 0.0022
Epoch 141/200, Loss: 0.0018
Epoch 161/200, Loss: 0.0014
Epoch 181/200, Loss: 0.0012
Evaluating the LSTM model:
R2 Score: 0.7037
MSE: 0.0125
MAPE: 9.0918


In [18]:
# print the comparison of the results
comparison_df = pd.DataFrame([rnn_results, lstm_results], index=["RNNModel", "LSTMModel"])

print("\nComparison of RNN and LSTM Models:\n")
print(comparison_df.round(4).to_string())


Comparison of RNN and LSTM Models:

           R2 Score     MSE    MAPE
RNNModel     0.5892  0.0174  9.2863
LSTMModel    0.7037  0.0125  9.0918
